<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/4%20-%20Tokens%2C%20Context%20%26%20Model%20Limitations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧮 Tokens, Context & Model Limitations

Every LLM has rules it can never break, no matter how good your prompt is: it reads/writes in fixed-size chunks called **tokens**, it can only "see" a limited stretch of text at once (the **context window**), and it knows nothing after its training cutoff. This notebook makes all three rules visible and runnable.

> ⚠️ **Demos 1–2 run anywhere** (only need `tiktoken`, no API key or running model).
> **Demos 3 onward run locally against LM Studio** (`http://localhost:1234/v1`) — they won't run unmodified in Colab.

**You'll do in code:** count tokens and estimate cost, overflow a tiny context window on purpose, tell `max_tokens` apart from the context window, reproduce a real overflow against a live model, implement sliding-window trimming and summarization, and see statelessness/knowledge-cutoff live.

## 📖 What is a token?

A **token** is the atomic unit a model actually reads, writes, and is billed on — never a whole word, never a raw character. A subword tokenizer (commonly BPE) breaks *any* input, including words it never saw in training, into chunks from a fixed vocabulary — so even a made-up word like "Dineshification" just splits into familiar pieces ("Din" + "esh" + "ification"), the way you'd sound out an unfamiliar name syllable by syllable.

Rule of thumb for English: **1 token ≈ ¾ of a word** — but this drifts a lot for code, numbers, emoji, and non-English text, as the demo below shows.

## 🔧 Setup — Install a Tokenizer

[`tiktoken`](https://github.com/openai/tiktoken) needs no API key or server — it just turns text into token IDs, so it's the fastest way to *see* tokens directly.

In [ ]:
%pip install -q tiktoken

: 

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def show_tokens(text):
    # Print how many tokens a string costs, and what each token decodes back to.
    tokens = encoding.encode(text)
    print(f"Text:   {text!r}")
    print(f"Words:  {len(text.split())}")
    print(f"Tokens: {len(tokens)}")
    pieces = [encoding.decode([t]) for t in tokens]
    print("Pieces:", pieces)
    print()

show_tokens("The cat sat on the mat.")
show_tokens("Dineshification of AI bootcamps is unstoppable!")

## 🧪 Demo — Tokens Are Not Words: Four Surprises

Four ways to trip up a "1 token = 1 word" assumption. Watch the token count relative to the word count each time.

In [ ]:
print("1) Plain English (close to 1 token per word)")
show_tokens("I want to learn how AI models work.")

print("2) Repeated punctuation and spacing (extra tokens for extra characters)")
show_tokens("Wait...       what?!")

print("3) A number, which often splits into multiple tokens")
show_tokens("The total was 3,482,910 rupees.")

print("4) Non-English text and an emoji (usually MORE tokens per visible character)")
show_tokens("नमस्ते, आप कैसे हैं? 🙏")

Cases 3 and 4 matter most: a single visible number or emoji can quietly cost several tokens. A paragraph that "looks short" in one language or format can be meaningfully more expensive than one that looks the same length in another.

## 📖 Why token count matters

Two independent things are both measured in tokens:

- **Cost** — providers charge separate `$ / 1M input tokens` and `$ / 1M output tokens` rates (output usually costs more, since generating each token needs a full forward pass). Rates change over time — check your provider's current pricing; the code below is for building intuition about *relative* cost, not quoting a live price.
- **Capacity** — the context window (next section) is also measured in tokens, so the same count that drives your bill also determines whether a request fits at all.

In [ ]:
def estimate_cost(input_text, expected_output_tokens, price_per_million_input, price_per_million_output):
    # Rough cost estimate for one request. Prices are $ per 1,000,000 tokens --
    # pass in whatever your provider currently lists; this function only does the arithmetic.
    input_tokens = len(encoding.encode(input_text))
    input_cost = input_tokens / 1_000_000 * price_per_million_input
    output_cost = expected_output_tokens / 1_000_000 * price_per_million_output
    print(f"Input tokens:  {input_tokens}")
    print(f"Output tokens: {expected_output_tokens} (assumed)")
    print(f"Estimated cost: ${input_cost + output_cost:.6f}  "
          f"(${input_cost:.6f} in + ${output_cost:.6f} out)")

# Example prices only -- swap these for your provider's current published rates.
EXAMPLE_PRICE_PER_MILLION_INPUT = 3.00
EXAMPLE_PRICE_PER_MILLION_OUTPUT = 15.00

long_document = ("Please summarize the following quarterly report. " * 200)
estimate_cost(long_document, expected_output_tokens=150,
              price_per_million_input=EXAMPLE_PRICE_PER_MILLION_INPUT,
              price_per_million_output=EXAMPLE_PRICE_PER_MILLION_OUTPUT)

## 📖 What is a context window?

The **context window** is the max number of tokens a model can process in one request — system prompt + every resent prior turn + the current message, combined. It's a hard architectural limit, not a setting, and it's not "memory" in a human sense: an LLM remembers nothing between separate API calls. A chat app *looks* like it remembers because it resends the entire conversation history on every new message — until that history stops fitting, and something has to be cut.

Context windows vary a lot by model (from a few thousand tokens on old/small models to over a million on long-context ones) and change as providers ship new versions — always check current docs rather than assuming a number.

## 🧪 Demo — Filling a Context Window and Watching It Overflow

A growing conversation against a **deliberately tiny** context window (60 tokens), so the overflow is visible without thousands of messages. The mechanism is identical at 60 or 200,000 tokens — only the numbers change.

In [ ]:
history = [
    "system: You are a helpful bootcamp assistant.",
    "user: Hi, I'm building a 5-day AI bootcamp.",
    "assistant: That's exciting! What's the focus of Day 1?",
    "user: LLM fundamentals: tokens, context windows, and prompting.",
    "assistant: Great foundation. Are the sessions hands-on with code?",
    "user: Yes, every theory section has a runnable demo.",
    "assistant: Nice, that keeps it concrete for beginners.",
    "user: Exactly. Now, quick one: what's the capital of France?",
]

CONTEXT_WINDOW_TOKENS = 60  # deliberately tiny, so the overflow is visible

def count_tokens(text):
    return len(encoding.encode(text))

def fits_in_window(messages, limit):
    # Return (kept_messages, dropped_messages, tokens_used) -- newest turns are kept first.
    kept, total = [], 0
    for message in reversed(messages):
        cost = count_tokens(message)
        if total + cost > limit:
            break
        kept.append(message)
        total += cost
    kept.reverse()
    dropped = messages[: len(messages) - len(kept)]
    return kept, dropped, total

kept, dropped, total = fits_in_window(history, CONTEXT_WINDOW_TOKENS)

print(f"Context window limit: {CONTEXT_WINDOW_TOKENS} tokens\n")
print("Dropped -- the model can no longer see these at all:")
for m in dropped:
    print("  x", m)

print("\nKept -- this is the ENTIRE reality the model has for this reply:")
for m in kept:
    print("  v", m)

print(f"\nTokens used: {total}/{CONTEXT_WINDOW_TOKENS}")

Notice what got dropped: the message that told the assistant what's being built. If the conversation continued and you asked "what am I building, again?", the model couldn't answer — that fact is no longer anywhere in what it's allowed to read.

## 🔌 Setup — Connect to LM Studio

The remaining demos call a real model.

In [ ]:
%pip install -q openai

In [ ]:
from openai import OpenAI

BASE_URL = "http://localhost:1234/v1"
client = OpenAI(base_url=BASE_URL, api_key="lm-studio")  # key is required by the SDK but ignored by LM Studio

models = client.models.list()
chat_models = [m.id for m in models.data if "embed" not in m.id.lower()]
if not chat_models:
    raise RuntimeError("No chat model found. Load one in LM Studio's Developer tab and start the server.")

MODEL = chat_models[0]
print(f"✅ Using MODEL: {MODEL}")

In [ ]:
def ask(prompt, system=None, max_tokens=200, temperature=0.0):
    # Send one prompt to the local model and print the reply. Used throughout this notebook.
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=temperature,
    )
    print(response.choices[0].message.content.strip())
    return response

## 📖 Context window vs. `max_tokens` — two different limits

- **Context window** — the model's fixed total capacity (input + output combined). You can't raise it; exceeding it is a hard API error or forced input truncation.
- **`max_tokens`** — a per-request parameter *you* set, capping only how many tokens the model may generate. Set it too low and it doesn't error — it silently truncates the *output*, often mid-sentence, because the model can't know in advance how long its answer will be.

Rule of thumb: an error, or old turns vanishing, points at the context window; an answer that just stops abruptly points at `max_tokens`.

In [ ]:
print("❌ max_tokens too small -- answer gets cut off mid-thought")
print("-" * 60)
ask("Explain what a context window is, in 4 full sentences.", max_tokens=15)

print("\n✅ Same prompt, enough room to actually finish")
print("-" * 60)
ask("Explain what a context window is, in 4 full sentences.", max_tokens=200)

## 🧪 Demo — Reproducing a Real Context-Window Overflow

This time no simulation — we build a genuinely long conversation and resend the **entire, untrimmed** history every turn, like a naive chatbot would, until the API rejects it or (for providers that silently truncate) the model starts acting as if the earliest turns never happened.

In [ ]:
def naive_chat_turn(all_messages, new_user_message, max_tokens=100):
    # The WRONG way to manage a long conversation: always resend everything, and hope it fits.
    all_messages.append({"role": "user", "content": new_user_message})
    try:
        response = client.chat.completions.create(
            model=MODEL, messages=all_messages, max_tokens=max_tokens, temperature=0.0,
        )
        reply = response.choices[0].message.content.strip()
        all_messages.append({"role": "assistant", "content": reply})
        print(f"[history so far: {sum(count_tokens(m['content']) for m in all_messages)} tokens]")
        print("assistant:", reply)
        return all_messages
    except Exception as e:
        print(f"💥 Request failed -- this is the context window being enforced for real:\n{e}")
        return all_messages

conversation = [{"role": "system", "content": "You are a concise assistant."}]

# Keep pushing large chunks of text into the conversation until something gives.
padding_paragraph = (
    "Here is some filler context to grow our conversation on purpose: "
    "the history of tokenization, subword units, and byte-pair encoding is long and detailed. " * 30
)

for turn in range(6):
    print(f"\n--- Turn {turn + 1} ---")
    conversation = naive_chat_turn(conversation, padding_paragraph + f" (turn {turn + 1}) Reply with just OK.")

> If your local model has a large context window, this loop might finish all 6 turns without failing — increase `range(6)` or lengthen `padding_paragraph` until it fails or starts ignoring early turns. The exact breaking point doesn't matter — the point is proving the limit is real.

## 🛠️ Living within a limited context window

Three strategies for a context budget smaller than your data:

- **Sliding-window truncation** — keep only the most recent `N` turns/tokens, drop the rest. Cheap, lossy; fine when only recent context matters.
- **Summarization** — periodically replace older turns with a short model-generated summary, keeping the gist at a fraction of the token cost.
- **Retrieval (RAG)** — store the full corpus externally and, per request, retrieve only the chunks relevant to the current question. The standard approach when total material vastly exceeds any context window (needs an embeddings + index step — its own topic).

The code below implements the first two.

In [ ]:
def trim_to_budget(messages, budget_tokens, keep_system=True):
    # Sliding-window strategy: keep the system message (if any) plus as many of the
    # most recent turns as fit in budget_tokens. Oldest non-system turns are dropped first.
    system_msgs = [m for m in messages if m["role"] == "system"] if keep_system else []
    other_msgs = [m for m in messages if m["role"] != "system"]

    used = sum(count_tokens(m["content"]) for m in system_msgs)
    kept = []
    for m in reversed(other_msgs):
        cost = count_tokens(m["content"])
        if used + cost > budget_tokens:
            break
        kept.append(m)
        used += cost
    kept.reverse()
    return system_msgs + kept, used


def summarize_and_trim(messages, keep_recent=2):
    # Summarization strategy: compress everything except the system message and the
    # most recent `keep_recent` turns into one short summary message, then keep that instead.
    system_msgs = [m for m in messages if m["role"] == "system"]
    other_msgs = [m for m in messages if m["role"] != "system"]

    if len(other_msgs) <= keep_recent:
        return messages, sum(count_tokens(m["content"]) for m in messages)

    to_summarize, recent = other_msgs[:-keep_recent], other_msgs[-keep_recent:]
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in to_summarize)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content":
                   f"Summarize the key facts from this conversation in 2 short sentences:\n\n{transcript}"}],
        max_tokens=80, temperature=0.0,
    )
    summary = response.choices[0].message.content.strip()
    summary_msg = {"role": "system", "content": f"Earlier conversation summary: {summary}"}

    rebuilt = system_msgs + [summary_msg] + recent
    return rebuilt, sum(count_tokens(m["content"]) for m in rebuilt)


# Try both strategies on the same oversized conversation.
long_conversation = [{"role": "system", "content": "You are a helpful assistant."}] + [
    {"role": "user" if i % 2 == 0 else "assistant", "content": f"Message {i}: " + padding_paragraph}
    for i in range(8)
]

full_size = sum(count_tokens(m["content"]) for m in long_conversation)
print(f"Original conversation: {full_size} tokens, {len(long_conversation)} messages\n")

BUDGET = 2000
trimmed, trimmed_size = trim_to_budget(long_conversation, budget_tokens=BUDGET)
print(f"After sliding-window trim to a {BUDGET}-token budget: {trimmed_size} tokens, "
      f"{len(trimmed)} messages kept\n")

summarized, summarized_size = summarize_and_trim(long_conversation, keep_recent=2)
print(f"After summarization (system + summary + last 2 turns): "
      f"{summarized_size} tokens, {len(summarized)} messages kept")
for m in summarized:
    label = m["content"][:80] + ("..." if len(m["content"]) > 80 else "")
    print(f"  [{m['role']}] {label}")

Sliding-window trimming keeps recent turns *verbatim* but throws older ones away completely. Summarization keeps a compressed trace of everything, at the cost of an extra model call. Pick based on whether you need exact old details (use retrieval instead) or just the gist (summarization is enough).

## 📖 Core model limitations (not bugs — structure)

An LLM is a fixed set of weights, frozen at the end of training, computing a probability distribution over the next token given only what's in its context window. Every limitation below follows from that one fact:

- 📅 **Knowledge cutoff** — nothing after the training cutoff could be in the data, and the model can't always tell "I wasn't trained on this" from "this doesn't exist."
- 🧠 **Statelessness** — weights don't change between requests, so nothing persists across turns except what you explicitly resend in the prompt.
- 🎲 **Non-determinism** — generation samples from a probability distribution rather than always taking the top token, so the same prompt can yield different (both reasonable) answers.
- 🎭 **Hallucination** — the model's objective is "produce a plausible continuation," not "produce a verified-true statement" — there's no built-in fact-check.
- 🧮 **Uneven reasoning** — arithmetic and multi-step logic need reliably executing a fixed procedure; next-token prediction is a learned approximation, not execution — which is why tool use (a real calculator/code interpreter) beats asking the model to "just compute it."

## 🧪 Demo — Seeing Two Limitations Live

In [ ]:
print("1) Statelessness -- a brand-new call has no memory of anything not resent")
print("-" * 60)
ask("What was the number I told you to remember?")
print("(There's no 'earlier' for this call -- nothing was ever sent, so there's nothing to recall.)")

print("\n2) Knowledge cutoff -- ask about something the model cannot possibly know")
print("-" * 60)
ask("In one sentence, who won the most recent championship of a league you'd have "
    "no way of knowing about after your training data ended? If you don't know, say so plainly.")

The first call proves statelessness: nothing "told to it earlier" in another cell or chat window exists here — it was never part of *this* prompt. The second nudges it to admit its cutoff rather than guess — compare to notebook 2's hallucination demo, where the question gave it no permission to say "I don't know."

## 🎯 Quick Reference

| Limit | What it caps | What happens if you exceed it | Fix |
|---|---|---|---|
| Context window | Input + output tokens, combined, per request | Hard error, or oldest content silently dropped | Sliding-window trim, summarization, or retrieval (RAG) |
| `max_tokens` | Output tokens only, per request (you set it) | Reply cuts off mid-sentence, no error | Raise the value, or ask for a shorter answer explicitly |
| Knowledge cutoff | What facts the model can know at all | Confident but wrong answer about recent events | Give it current facts in the prompt, or use a search-augmented model |
| Statelessness | What the model remembers between calls | "It forgot" what you said earlier | Resend relevant history yourself, every time |
| Non-determinism | Whether the same prompt gives the same output | Inconsistent answers across runs | Lower `temperature` (0 = closest to deterministic) |

## 📝 Recap

| Concept | What you learned |
|---|---|
| Token | The atomic unit a model reads/writes/bills on — a subword piece, not a word or character |
| Token count ≠ word count | Punctuation, numbers, emoji, and non-English text often cost more tokens per visible character |
| Context window | The hard total-token limit per request, covering the whole resent conversation |
| Context overflow | A real, reproducible failure — hard error or old turns silently vanishing |
| `max_tokens` vs. context window | `max_tokens` caps output and truncates silently; the context window caps everything and errors or drops old turns |
| Sliding window / summarization | Two ways to fit a long conversation into a limited budget |
| Knowledge cutoff & statelessness | Nothing persists between calls except what's resent; nothing exists after training ended |
| Non-determinism & hallucination | Sampling means repeat runs can differ; the model optimizes for plausible text, not verified truth |

**Next:** `llm_fundamentals_multi_provider.ipynb` — a working chat loop that manages history across turns.

## 🏋️ Try It Yourself (optional)

Using the helpers defined above (`count_tokens`, `trim_to_budget`, `summarize_and_trim`, `ask`), try:

1. Pick three sentences: one in English, one containing a long number or a URL, and one in a language other than English. Run `show_tokens()` on all three and compare tokens-per-word. Which one surprised you most?
2. Write your own `padding_paragraph`-style long conversation and find the smallest `budget_tokens` value where `trim_to_budget` has to drop the *system* message's important instructions — then fix your call to `trim_to_budget(..., keep_system=True)` and confirm it no longer does.
3. Call `ask(...)` with `max_tokens=10` on a question that needs a long answer, then rewrite the *prompt itself* (not the `max_tokens` value) to explicitly ask for a one-sentence answer instead. Compare the two failure/success modes.